In [6]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

import requests
from bs4 import BeautifulSoup

def fetch_website_contents(url: str) -> str:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.text

def fetch_website_links(url: str):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    links = []
    for a in soup.find_all("a", href=True):
        links.append(a["href"])
    
    return links

print("Tudo pronto 🚀")

Tudo pronto 🚀


In [ ]:
from openai import OpenAI

MODEL = "llama3.2"

client = OpenAI(
    
)

In [5]:
links = fetch_website_links("https://converse.com.br/")
links

['/customer/account',
 'https://converse.com.br/cupons-descontos',
 'https://converse.com.br/customer/account/',
 'https://converse.com.br/wishlist/',
 'https://converse.com.br/customer/account/create/',
 '#contentarea',
 'https://converse.com.br/',
 '/sale-c',
 '#store.menu',
 'https://converse.com.br/feminino-c',
 'https://converse.com.br/feminino-c/chuck-taylor',
 'https://converse.com.br/feminino-c/chuck-taylor/classic-chuck-c',
 'https://converse.com.br/feminino-c/chuck-taylor/chuck-70-c',
 'https://converse.com.br/feminino-c/chuck-taylor/chucks-in-love-c',
 'https://converse.com.br/feminino-c/chuck-taylor/seasonalcolors-feminino-c',
 'https://converse.com.br/feminino-c/em-alta-c',
 'https://converse.com.br/feminino-c/em-alta-c/mais-vendidos-c',
 'https://converse.com.br/feminino-c/em-alta-c/lancamentos-c',
 'https://converse.com.br/feminino-c/em-alta-c/exclusivos-do-site-c',
 'https://converse.com.br/feminino-c/em-alta-c/wave-motion-c',
 'https://converse.com.br/feminino-c/em-alt

In [18]:
link_system_prompt = """
You are provided with a list of links found on a webpage.

Select the most relevant links for a company brochure (e.g., About, Careers, Company pages).

You MUST follow these rules:
- Return ONLY valid JSON
- Do NOT include any text before or after the JSON
- Use EXACTLY this format:

{
  "links": [
    {"type": "about", "url": "https://example.com/about"},
    {"type": "careers", "url": "https://example.com/careers"}
  ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt("https://converse.com.br/"))


Here is the list of links on the website https://converse.com.br/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/customer/account
https://converse.com.br/cupons-descontos
https://converse.com.br/customer/account/
https://converse.com.br/wishlist/
https://converse.com.br/customer/account/create/
#contentarea
https://converse.com.br/
/sale-c
#store.menu
https://converse.com.br/feminino-c
https://converse.com.br/feminino-c/chuck-taylor
https://converse.com.br/feminino-c/chuck-taylor/classic-chuck-c
https://converse.com.br/feminino-c/chuck-taylor/chuck-70-c
https://converse.com.br/feminino-c/chuck-taylor/chucks-in-love-c
https://converse.com.br/feminino-c/chuck-taylor/seasonalcolors-feminino-c
https://converse.com.br/feminino-c/em-alta-c
https://converse.com.br/feminino-c/em-alta-c/mais-vendidos-c
https://

In [20]:
def select_relevant_links(url):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [13]:
select_relevant_links("https://converse.com.br/")

{'name': 'Converse Brasil',
 'url': 'https://converse.com.br',
 'description': 'Converse Brasil foi lançada em 2019. O site oferece uma ampla gama de produtos, incluindo cano alto, cano baixo, plataformas, calçados femininos e infantis, entre outros.'}

In [21]:
import json
import re

def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ]
        
    )
    
    result = response.choices[0].message.content
    print("Resposta do modelo:\n", result) 
    
    
    try:
        data = json.loads(result)
    except:
        match = re.search(r"\{.*\}", result, re.DOTALL)
        data = json.loads(match.group()) if match else {}
    
    links = data.get("links", [])
    
    print(f"Found {len(links)} relevant links")
    return links

In [22]:
select_relevant_links("https://converse.com.br/")

Selecting relevant links for https://converse.com.br/ by calling llama3.2
Resposta do modelo:
 Parece que você está testando várias páginas e links do site da Converse. Aqui estão os resultados:

1. **Converse Brasil**: O site oficial da Converse no Brasil é disponível em [www.converse.com.br](http://www.converse.com.br).
2. **Canao Alto**: A série de canoes da Converse é uma das mais populares do brand, disponível em [www.converse.com.br/canalos-c](http://www.converse.com.br/canalos-c).
3. **CanoBaixo**: Outra linha popular da Converse, disponível em [www.converse.com.br/canalos-baixos](http://www.converse.com.br/canalos-baixos).
4. **Plataforma**: Uma das linhas mais icônicas da Converse, disponível em [www.converse.com.br/platamorfos](http://www.converse.com.br/platamorfos).
5. **Feminino**: A linha de calçados femininos da Converse é uma das mais populares do brand, disponível em [www.converse.com.br/feminino-c](http://www.converse.com.br/feminino-c).
6. **Sports**: A linha de calç

[]

Segundo passo

In [25]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [26]:
print(fetch_page_and_all_relevant_links("https://converse.com.br"))

Selecting relevant links for https://converse.com.br by calling llama3.2
Resposta do modelo:
 Aqui estão os links e páginas que podem ser úteis:

**Páginas gerais**

* https://www.tiktok.com/@converse_br (TikTok oficial)
* https://www.youtube.com/user/conversebr (YouTube oficial)
* https://www.facebook.com/converse/ (Facebook oficial)

**Produtos e colecionáveis**

* Converse Classic Chuck Taylor: https://converse.com.br/classic-chuck-taylor
* Converse Bege: https://converse.com.br/bege
* Converse Azul: https://consecutive.com.br/azul
* Converse Verde: https://converse.com.br/verde-c
* Converse Double Stack: https://converse.com.br/double-stack-c

**Infantil e feminino**

* Converse Infantil: https://converse.com.br/infantil-c/em-alta/disney-c
* Converse Feminino: https://converse.com.br/feminino-c/em-alta/caramelo-c
* Converse Trainers Femininos: https://converse.com.br/feminino-c/sport-style/trainer-c

**Limited Edição e Colecionáveis**

* Limited Edição X Shai: https://converse.com.

In [28]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [29]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [30]:
get_brochure_user_prompt("Converse", "https://converse.com.br")

Selecting relevant links for https://converse.com.br by calling llama3.2
Resposta do modelo:
 Parece que você forneceu uma lista de URLs e links para a loja online da Converse Brasil. Vou organizar os links em categorias para torná-los mais facilmente acessíveis:

**Categorias**

1. **Homepage**
	* https://converse.com.br/
	* https://www.facebook.com/converse/?brand_redir=733293250106813
	* https://www.instagram.com/converse_br/
	* https://www.tiktok.com/@converse_br
	* https://www.youtube.com/user/conversebr
2. **Produtos**
	* https://converse.com.br/feminino-c/em-alta/caramelo-c
	* https://converse.com.br/classic-chuck-c
	* https://converse.com.br/bege-c
	* https://converse.com.br/azul
	* https://converse.com.br/verde-c
3. **Infantil**
	* https://converse.com.br/infantil-c/em-alta/disney-c
	* https://converse.com.br/infantil-c/compre-por-idade-genero/bebe-idade-1-12-meses-c
4. **Limited Editions**
	* https://converse.com.br/limited-edition-c/comprar/ver-todos-c/todos-c/converse-x-sha

'\nYou are looking at a company called: Converse\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\n <!doctype html><html lang="pt"><head ><meta charset="utf-8"/><script type="text/javascript">(window.NREUM||(NREUM={})).init={privacy:{cookies_enabled:true},ajax:{deny_list:["bam.nr-data.net"]},feature_flags:["soft_nav"],distributed_tracing:{enabled:true}};(window.NREUM||(NREUM={})).loader_config={agentID:"976270636",accountID:"3182476",trustKey:"1322840",xpid:"VwcPU1JUDhAJUlFaAgMFUVw=",licenseKey:"NRJS-fdf2e986a1d35ebfc81",applicationID:"960852278",browserID:"976270636"};;/*! For license information please see nr-loader-spa-1.310.1.min.js.LICENSE.txt */\n(()=>{var e,t,r={384:(e,t,r)=>{"use strict";r.d(t,{NT:()=>a,US:()=>l,Zm:()=>c,bQ:()=>u,dV:()=>d,pV:()=>f});var n=r(6154),i=r(1863),s=r(944),o=r(1910);const a={beacon:"bam.nr-data.net",errorBeacon:"ba

In [35]:
def create_brochure(company_name, url):
    response = client.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [36]:
create_brochure("Converse", "https://converse.com.br")

Selecting relevant links for https://converse.com.br by calling llama3.2
Resposta do modelo:
 Parece que você está analisando o sitio web da Converse, uma marca de calçados e acessórios. Aqui estão algumas observações sobre o conteúdo do site:

1. O site parece ter uma estrutura clara e fácil de navegar, com seções de produto, como "Feminino", "Masculino" e "Infantil", que permitem aos usuários filtrar os produtos por gênero.
2. Há uma seção de "Limited Edition" que sugere que a Converse lançou coleções temporárias de produtos de moda com partners e designer.
3. O site também possui uma seção de "Sale" (desconto) onde é possível encontrar ofertas e promoções em produtos da marca.
4. Existem links para redes sociais, como Facebook, Instagram e TikTok, o que indica que a Converse está presente nas plataformas de mídia social.
5. O site inclui uma seção de "Dúvidas frequentes" que cobre tópicos como devoluções e trocas, meios de pagamento e coupons descontos.

É importante observar que o 

Converse
------------

Wearable Conscience Since 1908
------------------------------

Converse is a world-renowned footwear brand that has been inspiring self-expression and creativity for over a century. Our mission is to empower individuals to express their individuality through our iconicChuck Taylor All Star sneakers, as well as an array of stylish clothing and accessories.

Company Culture
----------------

At Converse, we value creativity, sustainability, and inclusivity. We foster a culture that promotes collaboration, innovation, and empathy. Our team of passionate professionals is committed to delivering products and experiences that are both functional and fashionable.

Our Values:

*   **Innovate and Experiment**: We encourage our employees to think outside the box and push the boundaries of what's possible.
*   **Sustainability**: We're dedicated to reducing our environmental footprint through eco-friendly practices and materials.
*   **Inclusivity**: We believe that everyone deserves to express themselves through our products, regardless of age, size, or background.

Customers
------------

Converse has a loyal following of fashion enthusiasts, musicians, artists, and athletes who appreciate our signature style. From music festivals to everyday life, Converse is the perfect companion for anyone who wants to make a statement.

Some of Our Key Customers Include:

*   Musicians (e.g., Taylor Swift, Justin Timberlake)
*   Fashion Brands (e.g., Supreme, Opening Ceremony)
*   Artists (e.g., Shepard Fairey, Banksy)

Careers
---------

At Converse, we're not just looking for employees – we're building a community of like-minded individuals who share our passion for creativity, sustainability, and self-expression. Join us and become part of the Chuck Taylor crew!

Some Of Our Current Vacancies Include:

*   Marketing Manager
*   Sustainability Coordinator
*   Graphic Designer

Come and join our dynamic team of Converse enthusiasts to create innovative products and experiences that inspire a new generation.

Contact Us
-------------

Ready to be part of the Converse family? Visit our careers page to explore current job openings or submit your application directly. We can't wait to meet you!

Email: \[insert email address]
Phone: \[insert phone number]
Website: \[insert website URL]

In [37]:

def stream_brochure(company_name, url):
    stream = client.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("Converse", "https://converse.com.br")

Selecting relevant links for https://converse.com.br by calling llama3.2
Resposta do modelo:
 A página da Converse Brasil!

Aqui estão algumas observações sobre o conteúdo da página:

1. **Conteúdo principal**: A página é estruturada em seções principais, incluindo:
	* "Todos os lançamentos" com links para diferentes categorias de calçados, acessórios e serviços.
	* "Feminino" com links para calçados, acessórios e serviços específicos para mulheres.
	* "Masculino" com links para calçados, acessórios e serviços específicos para homens.
	* "Infantil" com links para calçados, acessórios e serviços específicos para crianças.
	* "Limited Edition" com links para calçados e acessórios de coleções limitadas.
2. **Barras de navegação**: A página tem barras de navegação nos lados superior e inferior, que permitem acessar diferentes seções da página sem precisar scrollar.
3. **Linkagens**: As linkagens são bem estruturadas e fáceis de encontrar, o que facilita a navegação pela página.
4. **Imagen

BrochureWithGradio

In [1]:
import gradio as gr

In [3]:
brochure_system_prompt = """
You are a professional marketing assistant.
Create a concise, attractive brochure for the company.
Use markdown formatting.
"""

In [4]:
def create_brochure_stream(company_name, url):
    messages = [
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    ]

    stream = client.chat.completions.create(
        model="llama3.2",
        messages=messages,
        stream=True
    )

    result = ""

    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            result += delta
            yield result


interface = gr.Interface(
    fn=create_brochure_stream,
    inputs=[
        gr.Textbox(label="Nome da empresa"),
        gr.Textbox(label="URL da empresa")
    ],
    outputs="markdown",
    title="Gerador de Brochure com IA",
    flagging_mode="never"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\Vitor\AppData\Roaming\Python\Python311\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Vitor\AppData\Roaming\Python\Python311\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Vitor\AppData\Roaming\Python\Python311\site-packages\gradio\blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Vitor\AppData\Roaming\Python\Python311\site-packages\gradio\blocks.py", line 1646, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Vitor\AppData\Roaming\Python\Python311\site-packages\gradio\utils.py", line 865, in a